In [ ]:
from pyspark.sql.functions import regexp_extract, col
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pathlib import Path

AUDITORIA_SILVER = []

BRONZE_CAMARA = "/Volumes/workspace/pi_ii_bronze/camara"
BRONZE_SENADO = "/Volumes/workspace/pi_ii_bronze/senado"

def arquivos_entidade(base_path, entidade):
    # Os quatro dígitos evitam misturar proposições, autores e fallback.
    padrao_arquivo = f"*/{entidade}_[0-9][0-9][0-9][0-9].csv"
    arquivos = sorted(str(path) for path in Path(base_path).glob(padrao_arquivo))
    if not arquivos:
        raise FileNotFoundError(f"Nenhum arquivo anual encontrado para {entidade} em {base_path}")
    return arquivos


def ler_entidade_por_uf(base_path: str, entidade: str, prefixo_uf: str):
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("mode", "FAILFAST")
        .option("enforceSchema", "false")
        .option("delimiter", ";")
        .option("quote", "\"")
        .option("escape", "\"")
        .option("multiLine", "true")
        .csv(arquivos_entidade(base_path, entidade))
        .withColumn("_caminho", col("_metadata.file_path"))
        .withColumn("uf_particao", regexp_extract("_caminho", rf"/{prefixo_uf}/([a-z]{{2}})/", 1))
        .withColumn("ano_particao", regexp_extract("_caminho", rf"{entidade}_(\d{{4}})\.csv", 1))
    )
    return df

def ler_cadastro_por_uf(base_path: str, arquivo: str, prefixo_uf: str):
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("mode", "FAILFAST")
        .option("enforceSchema", "false")
        .option("delimiter", ";")
        .option("quote", "\"")
        .option("escape", "\"")
        .option("multiLine", "true")
        .csv(f"{base_path}/*/{arquivo}")
        .withColumn("_caminho", col("_metadata.file_path"))
        .withColumn("uf_particao", regexp_extract("_caminho", rf"/{prefixo_uf}/([a-z]{{2}})/", 1))
    )
    return df

df_deputados = ler_cadastro_por_uf(BRONZE_CAMARA, "deputados.csv", "camara")
df_despesas_camara = ler_entidade_por_uf(BRONZE_CAMARA, "despesas", "camara")
df_proposicoes = ler_entidade_por_uf(BRONZE_CAMARA, "proposicoes", "camara")
# Fallbacks vazios têm cabeçalho mínimo. Leia cada arquivo pelo próprio cabeçalho.
for arquivo in arquivos_entidade(BRONZE_CAMARA, "proposicoes_fallback"):
    fallback = (spark.read.option("header", "true").option("inferSchema", "false")
        .option("mode", "FAILFAST").option("enforceSchema", "false")
        .option("delimiter", ";").option("quote", '\"').option("escape", '\"')
        .option("multiLine", "true").csv(arquivo))
    if "id" not in fallback.columns:
        if fallback.limit(1).count():
            raise ValueError(f"Fallback sem coluna id: {arquivo}")
        continue
    fallback = (fallback.withColumn("_caminho", F.lit(arquivo))
        .withColumn("uf_particao", F.regexp_extract(F.lit(arquivo), r"/camara/([a-z]{2})/", 1))
        .withColumn("ano_particao", F.regexp_extract(F.lit(arquivo), r"_(\d{4})\.csv$", 1)))
    fallback = fallback.join(df_proposicoes.select("id").distinct(), "id", "left_anti")
    df_proposicoes = df_proposicoes.unionByName(fallback, allowMissingColumns=True)

df_proposicoes_autores = ler_entidade_por_uf(BRONZE_CAMARA, "proposicoes_autores", "camara")
df_votacoes_camara = ler_entidade_por_uf(BRONZE_CAMARA, "votacoes", "camara")
df_votos_camara = ler_entidade_por_uf(BRONZE_CAMARA, "votos", "camara")

df_senadores = ler_cadastro_por_uf(BRONZE_SENADO, "senadores.csv", "senado")
df_senadores_exercicios = ler_cadastro_por_uf(BRONZE_SENADO, "senadores_exercicios.csv", "senado")
df_despesas_senado = ler_entidade_por_uf(BRONZE_SENADO, "despesas", "senado")
df_materias = ler_entidade_por_uf(BRONZE_SENADO, "materias", "senado")
df_materias_autorias = ler_entidade_por_uf(BRONZE_SENADO, "materias_autorias", "senado")
df_votacoes_senado = ler_entidade_por_uf(BRONZE_SENADO, "votacoes", "senado")
df_votos_senado = ler_entidade_por_uf(BRONZE_SENADO, "votos", "senado")


def gravar_silver(df, entidade, chaves):
    # Uma cópia por UF é esperada; versões conflitantes da mesma chave não são.
    dados = df.drop("uf_particao", "ano_particao", "_caminho").dropDuplicates()
    chave_invalida = F.lit(False)
    for chave in chaves:
        valor = F.col(chave)
        texto_vazio = F.trim(valor.cast("string")) == ""
        chave_invalida = chave_invalida | valor.isNull() | texto_vazio
    if dados.where(chave_invalida).limit(1).count():
        raise ValueError(f"{entidade}: chave vazia ou inválida; revise a Bronze antes de continuar.")
    if dados.groupBy(*chaves).count().where("count > 1").limit(1).count():
        raise ValueError(f"{entidade}: versões conflitantes da mesma chave; nenhuma versão foi escolhida.")

    # A partição de uma cópia não representa autoria nem UF do evento.
    consolidado = df.dropDuplicates(chaves)
    entrada = df.count()
    saida = consolidado.count()
    AUDITORIA_SILVER.append((entidade, entrada, saida, entrada - saida))
    consolidado.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        f"workspace.pi_ii_silver.{entidade}"
    )

In [ ]:
from pyspark.sql.functions import to_date, to_timestamp, trim, upper, col, expr

df_proposicoes_silver = (
    df_proposicoes
    .select(
        trim(col("id")).alias("id_proposicao"),
        upper(trim(col("siglaTipo"))).alias("sigla_tipo"),
        expr("try_cast(numero as int)").alias("numero"),
        expr("try_cast(ano as int)").alias("ano_proposicao"),
        col("descricaoTipo").alias("descricao_tipo"),
        col("ementa").alias("ementa"),
        expr("try_cast(dataApresentacao as date)").alias("data_apresentacao"),
        expr("try_cast(ultimoStatus_dataHora as timestamp)").alias("ultimo_status_data_hora"),
        col("ultimoStatus_siglaOrgao").alias("ultimo_status_sigla_orgao"),
        col("ultimoStatus_descricaoTramitacao").alias("ultimo_status_descricao_tramitacao"),
        col("ultimoStatus_descricaoSituacao").alias("ultimo_status_descricao_situacao"),
        col("ultimoStatus_apreciacao").alias("ultimo_status_apreciacao"),
        col("urlInteiroTeor").alias("url_inteiro_teor"),
        col("uf_particao"),
        col("ano_particao"),
    )
)

gravar_silver(df_proposicoes_silver, "proposicoes", ['id_proposicao'])

In [ ]:
df_votos_camara_silver = (
    df_votos_camara
    .select(
        col("idVotacao").alias("id_votacao"),
        to_timestamp(col("dataHoraVoto")).alias("data_hora_voto"),
        upper(trim(col("voto"))).alias("voto"),
        trim(col("deputado_id")).alias("id_deputado"),
        trim(col("deputado_nome")).alias("nome_deputado"),
        upper(trim(col("deputado_siglaPartido"))).alias("partido"),
        upper(trim(col("deputado_siglaUf"))).alias("uf_deputado"),
        col("deputado_idLegislatura").cast("int").alias("id_legislatura"),
        col("uf_particao"),
        col("ano_particao"),
    )
)

gravar_silver(df_votos_camara_silver, "votos_camara", ['id_votacao', 'id_deputado'])

In [ ]:
df_despesas_senado_silver = (
    df_despesas_senado
    .select(
        col("id").alias("id_despesa"),
        trim(col("codSenador")).alias("id_senador"),
        trim(col("nomeSenador")).alias("nome_senador"),
        col("tipoDespesa").alias("tipo_despesa"),
        trim(col("fornecedor")).alias("fornecedor"),
        col("cpfCnpj").alias("cnpj_cpf_fornecedor"),
        to_date(col("data")).alias("data_despesa"),
        col("ano").cast("int").alias("ano"),
        col("mes").cast("int").alias("mes"),
        col("valorReembolsado").cast("decimal(12,2)").alias("valor_reembolsado"),
        upper(trim(col("_uf_recorte"))).alias("uf"),
        col("uf_particao"),
        col("ano_particao"),
    )
)
if df_despesas_senado_silver.where("valor_reembolsado IS NULL OR ano IS NULL OR mes NOT BETWEEN 1 AND 12 OR mes IS NULL").limit(1).count():
    raise ValueError("Despesas do Senado: valor reembolsado ou competência inválidos.")

gravar_silver(df_despesas_senado_silver, "despesas_senado", ['id_despesa'])

In [ ]:
# Não existe ID oficial de movimento no CSV. O hash da linha bruta mais a
# ocorrência preserva inclusive linhas idênticas; ideDocumento continua disponível.
for campo in ["numParcela", "numRessarcimento", "numLote", "vlrRestituicao", "datPagamentoRestituicao"]:
    if campo not in df_despesas_camara.columns:
        df_despesas_camara = df_despesas_camara.withColumn(campo, F.lit(None).cast("string"))
colunas_origem = sorted(
    nome for nome in df_despesas_camara.columns
    if not nome.startswith("_") and nome not in {"uf_particao", "ano_particao"}
)
campos_origem = F.struct(*[F.col(nome) for nome in colunas_origem])
conteudo_linha = F.to_json(campos_origem, {"ignoreNullFields": "false"})
linhas_com_mesmo_conteudo = Window.partitionBy("_hash_movimento").orderBy(F.lit(1))

df_movimentos = df_despesas_camara.withColumn(
    "_hash_movimento", F.sha2(conteudo_linha, 256)
).withColumn(
    "_ocorrencia", F.row_number().over(linhas_com_mesmo_conteudo)
)
df_despesas_camara_silver = (
    df_movimentos
    .select(
        F.concat_ws(":", F.lit("ceap"), F.col("_hash_movimento"), F.col("_ocorrencia")).alias("id_movimento"),
        trim(col("ideDocumento")).alias("id_documento"),
        trim(col("ideCadastro")).alias("id_deputado_api"),
        col("numParcela").alias("parcela"),
        col("numRessarcimento").alias("ressarcimento"),
        col("numLote").alias("lote_documentos"),
        col("vlrRestituicao").cast("decimal(12,2)").alias("valor_restituicao"),
        col("datPagamentoRestituicao").alias("data_restituicao"),
        col("nuDeputadoId").alias("id_deputado"),
        trim(col("txNomeParlamentar")).alias("nome_parlamentar"),
        upper(trim(col("sgUF"))).alias("uf"),
        upper(trim(col("sgPartido"))).alias("partido"),
        col("numAno").cast("int").alias("ano"),
        col("numMes").cast("int").alias("mes"),
        col("txtDescricao").alias("tipo_despesa"),
        trim(col("txtFornecedor")).alias("fornecedor"),
        col("txtCNPJCPF").alias("cnpj_cpf_fornecedor"),
        to_date(col("datEmissao")).alias("data_emissao"),
        col("vlrDocumento").cast("decimal(12,2)").alias("valor_documento"),
        col("vlrGlosa").cast("decimal(12,2)").alias("valor_glosa"),
        col("vlrLiquido").cast("decimal(12,2)").alias("valor_liquido"),
        col("_caminho"),
        col("uf_particao"),
        col("ano_particao"),
    )
)
if df_despesas_camara_silver.where("valor_liquido IS NULL OR ano IS NULL OR mes NOT BETWEEN 1 AND 12 OR mes IS NULL").limit(1).count():
    raise ValueError("Despesas da Câmara: valor líquido ou competência inválidos.")
assert df_movimentos.count() == df_despesas_camara_silver.count(), "Movimentos perdidos na transformação"
total_bronze = df_movimentos.select(F.sum(F.col("vlrLiquido").cast("decimal(12,2)"))).first()[0]
total_silver = df_despesas_camara_silver.select(F.sum("valor_liquido")).first()[0]
assert total_bronze == total_silver, "Valor líquido divergente entre Bronze e Silver"

# Um código CEAP não pode apontar para dois IDs de API.
if df_despesas_camara_silver.where("id_deputado_api IS NOT NULL AND id_deputado_api <> ''").groupBy("id_deputado").agg(
    F.countDistinct("id_deputado_api").alias("ids_api")
).where("ids_api > 1").limit(1).count():
    raise ValueError("Código CEAP com mais de um ideCadastro; revise a fonte.")

gravar_silver(df_despesas_camara_silver, "despesas_camara", ['id_movimento'])

In [ ]:
df_materias_silver = (
    df_materias
    .select(
        trim(col("Materia_Codigo")).alias("id_materia"),
        col("Materia_Sigla").alias("sigla"),
        col("Materia_Numero").cast("int").alias("numero"),
        col("Materia_Ano").cast("int").alias("ano_materia"),
        col("Materia_Ementa").alias("ementa"),
        to_date(col("Materia_Data")).alias("data_materia"),
        col("IndicadorAutorPrincipal").alias("indicador_autor_principal"),
        col("IndicadorOutrosAutores").alias("indicador_outros_autores"),
        trim(col("_codigo_parlamentar_consulta")).alias("id_senador"),
        upper(trim(col("_uf_recorte"))).alias("uf"),
        col("_registro_valido_leg57").cast("boolean").alias("registro_valido_leg57"),
        col("uf_particao"),
        col("ano_particao"),
    )
)

gravar_silver(df_materias_silver, "materias", ['id_materia', 'id_senador'])

In [ ]:
from pyspark.sql.functions import to_date, to_timestamp, trim, upper, col, expr

df_deputados_silver = (
    df_deputados
    .select(
        trim(col("id")).alias("id_deputado"),
        trim(col("nome")).alias("nome"),
        upper(trim(col("siglaPartido"))).alias("partido"),
        upper(trim(col("siglaUf"))).alias("uf"),
        expr("try_cast(idLegislatura as int)").alias("id_legislatura"),
        col("urlFoto").alias("url_foto"),
        col("email").alias("email"),
        col("uf_particao"),
    )
)
gravar_silver(df_deputados_silver, "deputados", ['id_deputado'])

In [ ]:
df_senadores_silver = (
    df_senadores
    .select(
        trim(col("codigo_parlamentar")).alias("id_senador"),
        trim(col("nome_parlamentar")).alias("nome"),
        upper(trim(col("partido"))).alias("partido"),
        upper(trim(col("uf"))).alias("uf"),
        col("primeiro_exercicio_leg57").alias("primeiro_exercicio_leg57"),
        col("ultimo_fim_exercicio_leg57").alias("ultimo_fim_exercicio_leg57"),
        col("participacoes").alias("participacoes"),
        col("uf_particao"),
    )
)
gravar_silver(df_senadores_silver, "senadores", ['id_senador'])

In [ ]:
df_senadores_exercicios_silver = (
    df_senadores_exercicios
    .select(
        trim(col("codigo_exercicio")).alias("id_exercicio"),
        trim(col("codigo_parlamentar")).alias("id_senador"),
        trim(col("nome_parlamentar")).alias("nome"),
        upper(trim(col("partido"))).alias("partido"),
        upper(trim(col("uf"))).alias("uf"),
        trim(col("codigo_mandato")).alias("id_mandato"),
        col("participacao").alias("participacao"),
        col("data_inicio_exercicio").alias("data_inicio_exercicio"),
        col("data_fim_exercicio").alias("data_fim_exercicio"),
        col("sigla_causa_afastamento").alias("sigla_causa_afastamento"),
        col("descricao_causa_afastamento").alias("descricao_causa_afastamento"),
        col("uf_particao"),
    )
)
gravar_silver(df_senadores_exercicios_silver, "senadores_exercicios", ['id_exercicio'])

In [ ]:
df_proposicoes_autores_silver = (
    df_proposicoes_autores
    .select(
        trim(col("idProposicao")).alias("id_proposicao"),
        trim(col("idDeputadoAutor")).alias("id_deputado_autor"),
        col("tipoAutor").alias("tipo_autor"),
        trim(col("nomeAutor")).alias("nome_autor"),
        upper(trim(col("siglaPartidoAutor"))).alias("partido_autor"),
        upper(trim(col("siglaUFAutor"))).alias("uf_autor"),
        expr("try_cast(ordemAssinatura as int)").alias("ordem_assinatura"),
        expr("try_cast(proponente as int)").alias("proponente"),
        col("uf_particao"),
        col("ano_particao"),
    )
)
gravar_silver(df_proposicoes_autores_silver, "proposicoes_autores", ['id_proposicao', 'id_deputado_autor'])

In [ ]:
df_materias_autorias_silver = (
    df_materias_autorias
    .select(
        trim(col("Materia_Codigo")).alias("id_materia"),
        col("Materia_Sigla").alias("sigla"),
        expr("try_cast(Materia_Numero as int)").alias("numero"),
        expr("try_cast(Materia_Ano as int)").alias("ano_materia"),
        trim(col("_codigo_parlamentar_consulta")).alias("id_senador"),
        col("IndicadorAutorPrincipal").alias("indicador_autor_principal"),
        col("IndicadorOutrosAutores").alias("indicador_outros_autores"),
        upper(trim(col("_uf_recorte"))).alias("uf"),
        col("_registro_valido_leg57").cast("boolean").alias("registro_valido_leg57"),
        col("uf_particao"),
        col("ano_particao"),
    )
)
gravar_silver(df_materias_autorias_silver, "materias_autorias", ['id_materia', 'id_senador'])

In [ ]:
df_votacoes_camara_silver = (
    df_votacoes_camara
    .select(
        col("id").alias("id_votacao"),
        to_date(col("data")).alias("data_votacao"),
        col("dataHoraRegistro").alias("data_hora_registro"),
        upper(trim(col("siglaOrgao"))).alias("sigla_orgao"),
        expr("try_cast(aprovacao as int)").alias("aprovacao"),
        expr("try_cast(votosSim as int)").alias("votos_sim"),
        expr("try_cast(votosNao as int)").alias("votos_nao"),
        expr("try_cast(votosOutros as int)").alias("votos_outros"),
        col("descricao").alias("descricao"),
        expr("try_cast(ultimaApresentacaoProposicao_idProposicao as int)").alias("id_proposicao"),
        col("uf_particao"),
        col("ano_particao"),
    )
)
gravar_silver(df_votacoes_camara_silver, "votacoes_camara", ['id_votacao'])

In [ ]:
df_votacoes_senado_silver = (
    df_votacoes_senado
    .select(
        trim(col("CodigoSessaoVotacao")).alias("id_votacao"),
        trim(col("CodigoMateria")).alias("id_materia"),
        to_date(col("DataSessao")).alias("data_votacao"),
        col("HoraInicio").alias("hora_inicio"),
        col("Secreta").alias("secreta"),
        col("DescricaoVotacao").alias("descricao"),
        col("Resultado").alias("resultado"),
        expr("try_cast(TotalVotosSim as int)").alias("votos_sim"),
        expr("try_cast(TotalVotosNao as int)").alias("votos_nao"),
        expr("try_cast(TotalVotosAbstencao as int)").alias("votos_abstencao"),
        col("uf_particao"),
        col("ano_particao"),
    )
)
gravar_silver(df_votacoes_senado_silver, "votacoes_senado", ['id_votacao'])

In [ ]:
df_votos_senado_silver = (
    df_votos_senado
    .select(
        trim(col("_CodigoSessaoVotacao")).alias("id_votacao"),
        trim(col("CodigoParlamentar")).alias("id_senador"),
        trim(col("NomeParlamentar")).alias("nome_senador"),
        upper(trim(col("SiglaPartido"))).alias("partido"),
        upper(trim(col("SiglaUF"))).alias("uf_senador"),
        upper(trim(col("Voto"))).alias("voto"),
        col("DescricaoVoto").alias("descricao_voto"),
        col("_DataSessao").alias("data_votacao"),
        upper(trim(col("_uf_recorte"))).alias("uf"),
        col("uf_particao"),
        col("ano_particao"),
    )
)
gravar_silver(df_votos_senado_silver, "votos_senado", ['id_votacao', 'id_senador'])

In [ ]:
spark.createDataFrame(AUDITORIA_SILVER, "entidade string, entrada long, saida long, copias_repetidas long").write.format("delta").mode("overwrite").saveAsTable("workspace.pi_ii_silver.auditoria_carga")
display(spark.table("workspace.pi_ii_silver.auditoria_carga"))